# Module 10: Meta-Learning Recommender for Feature Selection

## Learning Objectives
By completing this notebook, you will:
1. Compute dataset meta-features (statistical, correlation, MI-based, landmarking) for 10+ OpenML-equivalent datasets
2. Run 5 feature selection methods on each dataset and record downstream performance
3. Train a meta-model that predicts the best selector from meta-features
4. Evaluate the meta-recommender using Leave-One-Dataset-Out cross-validation
5. Build a practical tool: "which selector should I use for my data?"

## Prerequisites
- Notebooks 01 and 02, and all three guides in this module
- Understanding of cross-validation and meta-learning concepts

## Estimated Time: 15 minutes

## Datasets
We use 12 datasets from sklearn (and synthetic variants) covering diverse characteristics:
low/high dimensionality, balanced/imbalanced classes, linear/nonlinear structure, correlated/independent features.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from itertools import combinations

from scipy import stats
from sklearn.datasets import (
    load_breast_cancer, load_wine, load_digits, load_iris,
    make_classification, make_friedman1
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
np.random.seed(42)

print("All imports successful.")

## 1. Load Diverse Datasets

We use 12 datasets with varying characteristics to build a rich meta-training set.
Each dataset has a name, (X, y) arrays, and a brief description.

In [ ]:
def load_all_datasets() -> list[dict]:
    """
    Load 12 diverse classification datasets for meta-learning.
    Returns list of dicts with keys: name, X, y, description.
    """
    datasets = []

    # 1. Breast Cancer (small, moderate p, correlated)
    bc = load_breast_cancer()
    datasets.append({'name': 'breast_cancer', 'X': bc.data, 'y': bc.target,
                      'desc': 'n=569, p=30, binary, correlated features'})

    # 2. Wine (small, low p, multiclass)
    w = load_wine()
    datasets.append({'name': 'wine', 'X': w.data, 'y': w.target,
                      'desc': 'n=178, p=13, multiclass, chemistry'})

    # 3. Iris (tiny, low p, multiclass)
    ir = load_iris()
    datasets.append({'name': 'iris', 'X': ir.data, 'y': ir.target,
                      'desc': 'n=150, p=4, multiclass, biological'})

    # 4. Digits (medium, higher p, multiclass)
    dig = load_digits()
    datasets.append({'name': 'digits', 'X': dig.data, 'y': dig.target,
                      'desc': 'n=1797, p=64, multiclass, image pixels'})

    # 5. High-dimensional, many noise features
    X5, y5 = make_classification(n_samples=800, n_features=100, n_informative=10,
                                   n_redundant=5, flip_y=0.05, random_state=1)
    datasets.append({'name': 'high_dim_noisy', 'X': X5, 'y': y5,
                      'desc': 'n=800, p=100, 10 informative, 90 noise'})

    # 6. Highly correlated features
    X6, y6 = make_classification(n_samples=600, n_features=30, n_informative=8,
                                   n_redundant=15,  # 15 redundant = highly correlated
                                   flip_y=0.03, random_state=2)
    datasets.append({'name': 'highly_correlated', 'X': X6, 'y': y6,
                      'desc': 'n=600, p=30, highly correlated features'})

    # 7. Imbalanced classes
    X7, y7 = make_classification(n_samples=1000, n_features=20, n_informative=8,
                                   weights=[0.9, 0.1],  # 90/10 imbalance
                                   flip_y=0.02, random_state=3)
    datasets.append({'name': 'imbalanced', 'X': X7, 'y': y7,
                      'desc': 'n=1000, p=20, 90/10 class imbalance'})

    # 8. Few samples, moderate features (small sample regime)
    X8, y8 = make_classification(n_samples=150, n_features=40, n_informative=12,
                                   n_redundant=5, flip_y=0.05, random_state=4)
    datasets.append({'name': 'small_sample', 'X': X8, 'y': y8,
                      'desc': 'n=150, p=40, small sample regime (n < 4p)'})

    # 9. Large n, clean signal
    X9, y9 = make_classification(n_samples=3000, n_features=25, n_informative=12,
                                   n_redundant=5, flip_y=0.01, random_state=5)
    datasets.append({'name': 'large_clean', 'X': X9, 'y': y9,
                      'desc': 'n=3000, p=25, large n, clean signal'})

    # 10. Very sparse signal (few informative, many noise)
    X10, y10 = make_classification(n_samples=1000, n_features=200, n_informative=5,
                                    n_redundant=2, flip_y=0.05, random_state=6)
    datasets.append({'name': 'sparse_signal', 'X': X10, 'y': y10,
                      'desc': 'n=1000, p=200, 5 informative (very sparse)'})

    # 11. Medium balanced, moderate correlation
    X11, y11 = make_classification(n_samples=1200, n_features=50, n_informative=15,
                                    n_redundant=10, flip_y=0.04, random_state=7)
    datasets.append({'name': 'medium_balanced', 'X': X11, 'y': y11,
                      'desc': 'n=1200, p=50, balanced, moderate correlation'})

    # 12. Very high-dimensional (challenging regime)
    X12, y12 = make_classification(n_samples=500, n_features=300, n_informative=8,
                                    n_redundant=10, flip_y=0.06, random_state=8)
    datasets.append({'name': 'very_high_dim', 'X': X12, 'y': y12,
                      'desc': 'n=500, p=300, extreme n << p regime'})

    return datasets


datasets = load_all_datasets()

print(f"Loaded {len(datasets)} datasets:\n")
print(f"{'Name':25s} {'n':>6s} {'p':>6s} {'n/p ratio':>12s} {'Description'}")
print("-" * 90)
for d in datasets:
    n, p = d['X'].shape
    print(f"{d['name']:25s} {n:6d} {p:6d} {n/p:12.1f}   {d['desc']}")

## 2. Compute Meta-Features for Each Dataset

Meta-features are cheap-to-compute properties that characterise the dataset. They are the input to our meta-model.

In [ ]:
def gini_coefficient(arr: np.ndarray) -> float:
    """Gini coefficient: concentration measure in [0, 1]."""
    arr = np.sort(np.abs(arr))
    n = len(arr)
    if n == 0 or arr.sum() == 0:
        return 0.0
    cumsum = np.cumsum(arr)
    return float((2 * np.sum(cumsum) / (n * cumsum[-1])) - (n + 1) / n)


def compute_meta_features(X: np.ndarray, y: np.ndarray, name: str) -> dict:
    """
    Compute all meta-features for a dataset.

    Covers four categories:
    1. Statistical (size, class distribution)
    2. Correlation structure (inter-feature redundancy)
    3. Information-theoretic (MI with target)
    4. Landmarking (proxy algorithm performance)
    """
    n, p = X.shape
    mf = {'dataset_name': name}

    # --- Category 1: Statistical ---
    mf['n_samples'] = n
    mf['n_features'] = p
    mf['log_n'] = float(np.log1p(n))
    mf['log_p'] = float(np.log1p(p))
    mf['log_ratio_n_p'] = float(np.log1p(n / p))

    # Target distribution
    unique_classes, counts = np.unique(y, return_counts=True)
    n_classes = len(unique_classes)
    mf['n_classes'] = n_classes
    mf['class_balance'] = float(counts.min() / counts.max())  # 1 = balanced
    mf['class_entropy'] = float(stats.entropy(counts / n))

    # Feature statistics
    mf['mean_feature_skew'] = float(np.mean(np.abs(stats.skew(X, axis=0))))
    mf['frac_near_constant'] = float(np.mean(X.std(axis=0) < 0.01))

    # --- Category 2: Correlation structure ---
    sample_p = min(p, 100)  # avoid O(p^2) bottleneck
    if p > 100:
        rng = np.random.default_rng(42)
        feat_idx = rng.choice(p, size=sample_p, replace=False)
        X_sample = X[:, feat_idx]
    else:
        X_sample = X

    corr = np.corrcoef(X_sample.T)
    np.fill_diagonal(corr, 0)
    abs_corr = np.abs(corr[np.triu_indices_from(corr, k=1)])

    mf['mean_abs_correlation'] = float(np.mean(abs_corr))
    mf['max_abs_correlation'] = float(np.max(abs_corr))
    mf['frac_highly_correlated'] = float(np.mean(abs_corr > 0.7))

    # Effective rank
    pca = PCA(n_components=min(50, sample_p, n - 1))
    pca.fit(X_sample)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n_95 = int(np.searchsorted(cumvar, 0.95)) + 1
    mf['effective_rank_ratio'] = float(n_95 / sample_p)

    # --- Category 3: Information-theoretic ---
    mi_sample_p = min(p, 50)
    rng = np.random.default_rng(42)
    mi_idx = rng.choice(p, size=mi_sample_p, replace=False)
    mi_scores = mutual_info_classif(X[:, mi_idx], y, random_state=42)

    mf['mean_mi'] = float(np.mean(mi_scores))
    mf['max_mi'] = float(np.max(mi_scores))
    mf['frac_zero_mi'] = float(np.mean(mi_scores < 0.01))
    mf['mi_gini'] = float(gini_coefficient(mi_scores))

    # --- Category 4: Landmarking ---
    max_n_land = min(n, 400)
    land_idx = np.random.RandomState(42).choice(n, size=max_n_land, replace=False)
    X_land = StandardScaler().fit_transform(X[land_idx])
    y_land = y[land_idx]

    cv_land = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    # Decision stump
    dt1 = DecisionTreeClassifier(max_depth=1, random_state=42)
    dt1_scores = cross_val_score(dt1, X_land, y_land, cv=cv_land,
                                  scoring='balanced_accuracy')
    mf['dt1_acc'] = float(dt1_scores.mean())

    # 1-NN
    knn1 = KNeighborsClassifier(n_neighbors=1)
    knn1_scores = cross_val_score(knn1, X_land, y_land, cv=cv_land,
                                   scoring='balanced_accuracy')
    mf['knn1_acc'] = float(knn1_scores.mean())

    # Baseline
    dummy = DummyClassifier(strategy='most_frequent')
    dummy_scores = cross_val_score(dummy, X_land, y_land, cv=cv_land,
                                    scoring='balanced_accuracy')
    mf['baseline_acc'] = float(dummy_scores.mean())

    mf['dt1_lift'] = mf['dt1_acc'] - mf['baseline_acc']
    mf['knn1_lift'] = mf['knn1_acc'] - mf['baseline_acc']

    return mf


print("Computing meta-features for all datasets...")
all_meta_features = []
for d in datasets:
    mf = compute_meta_features(d['X'], d['y'], d['name'])
    all_meta_features.append(mf)
    print(f"  {d['name']:25s}: n/p={d['X'].shape[0]/d['X'].shape[1]:.1f}, "
          f"mean_corr={mf['mean_abs_correlation']:.3f}, "
          f"frac_zero_mi={mf['frac_zero_mi']:.2f}, "
          f"dt1_lift={mf['dt1_lift']:.3f}")

meta_df = pd.DataFrame(all_meta_features)
print(f"\nMeta-feature matrix shape: {meta_df.shape}")

## 3. Run 5 Selectors on Each Dataset

For each dataset, we run 5 selectors and evaluate downstream performance with cross-validated GradientBoosting accuracy.

In [ ]:
def run_selector(selector_name: str, X: np.ndarray, y: np.ndarray,
                  n_select: int) -> list:
    """
    Run a named feature selector and return selected feature indices.

    Parameters
    ----------
    selector_name : 'mi' | 'lasso' | 'rf_importance' | 'rf_boruta_approx' | 'ensemble'
    n_select : number of features to select

    Returns
    -------
    list of selected feature indices
    """
    p = X.shape[1]
    n_select = min(n_select, p)

    if selector_name == 'mi':
        scores = mutual_info_classif(X, y, random_state=42)
        return np.argsort(-scores)[:n_select].tolist()

    elif selector_name == 'lasso':
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        lasso = LassoCV(cv=3, random_state=42, max_iter=5000, n_alphas=30)
        lasso.fit(X_scaled, y)
        coef_abs = np.abs(lasso.coef_)
        # Ensure we always get n_select features
        top_idx = np.argsort(-coef_abs)[:n_select]
        return top_idx.tolist()

    elif selector_name == 'rf_importance':
        rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        rf.fit(X, y)
        return np.argsort(-rf.feature_importances_)[:n_select].tolist()

    elif selector_name == 'rf_boruta_approx':
        # Boruta approximation: RF importance vs shadow feature importance
        # Shadow features = randomly permuted copies of real features
        rng = np.random.default_rng(42)
        X_shadow = rng.permutation(X)  # permute rows
        X_aug = np.hstack([X, X_shadow])
        rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        rf.fit(X_aug, y)
        imp = rf.feature_importances_
        real_imp = imp[:p]
        shadow_max = imp[p:].max()
        # Features that beat the best shadow feature
        confirmed = np.where(real_imp > shadow_max)[0]
        if len(confirmed) >= n_select:
            return np.argsort(-real_imp[confirmed])[:n_select].tolist()
        else:
            # Fall back to top by real importance
            return np.argsort(-real_imp)[:n_select].tolist()

    elif selector_name == 'elastic_net':
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        from sklearn.linear_model import ElasticNetCV
        enet = ElasticNetCV(l1_ratio=[0.5, 0.7, 0.9], cv=3,
                             random_state=42, max_iter=5000, n_alphas=20)
        enet.fit(X_scaled, y)
        coef_abs = np.abs(enet.coef_)
        return np.argsort(-coef_abs)[:n_select].tolist()

    else:
        raise ValueError(f'Unknown selector: {selector_name}')


SELECTOR_NAMES = ['mi', 'lasso', 'rf_importance', 'rf_boruta_approx', 'elastic_net']
N_SELECT = 10  # select top-10 features from each dataset

eval_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
eval_model = GradientBoostingClassifier(n_estimators=80, random_state=42)

print(f"Running {len(SELECTOR_NAMES)} selectors × {len(datasets)} datasets...")
print(f"(Selecting top-{N_SELECT} features, evaluated with GradientBoosting 5-fold CV)\n")

for d in datasets:
    selector_scores = {}
    X_d, y_d = d['X'], d['y']
    p_d = X_d.shape[1]
    n_sel = min(N_SELECT, p_d)

    for sel_name in SELECTOR_NAMES:
        selected = run_selector(sel_name, X_d, y_d, n_select=n_sel)
        X_sel = X_d[:, selected]
        cv_scores = cross_val_score(eval_model, X_sel, y_d,
                                     cv=eval_cv, scoring='balanced_accuracy')
        selector_scores[sel_name] = float(cv_scores.mean())

    d['selector_scores'] = selector_scores
    d['best_selector'] = max(selector_scores, key=selector_scores.get)
    d['best_score'] = selector_scores[d['best_selector']]

    best = d['best_selector']
    print(f"  {d['name']:25s}: best = {best:20s} ({selector_scores[best]:.4f}) | "
          f"scores: {' '.join([f'{s:.3f}' for s in selector_scores.values()])}")

print("\nAll evaluations complete.")

## 4. Build the Meta-Dataset

Combine meta-features with selector performance labels to create the training data for our meta-model.

In [ ]:
# Add best_selector and individual scores to meta_df
for d in datasets:
    idx = meta_df.index[meta_df['dataset_name'] == d['name']]
    meta_df.loc[idx, 'best_selector'] = d['best_selector']
    meta_df.loc[idx, 'best_score'] = d['best_score']
    for sel_name, score in d['selector_scores'].items():
        meta_df.loc[idx, f'score_{sel_name}'] = score

print("Meta-dataset summary:")
print(meta_df[['dataset_name', 'best_selector', 'best_score']].to_string(index=False))
print(f"\nClass distribution of best selectors:")
print(meta_df['best_selector'].value_counts().to_string())

## 5. Train the Meta-Model

A Random Forest classifier takes meta-features as input and predicts the best selector.

In [ ]:
META_FEATURE_COLS = [
    'log_n', 'log_p', 'log_ratio_n_p',
    'n_classes', 'class_balance', 'class_entropy',
    'mean_abs_correlation', 'max_abs_correlation', 'frac_highly_correlated',
    'effective_rank_ratio',
    'mean_mi', 'max_mi', 'frac_zero_mi', 'mi_gini',
    'dt1_lift', 'knn1_lift',
]

available_cols = [c for c in META_FEATURE_COLS if c in meta_df.columns]
X_meta = meta_df[available_cols].fillna(0).values

le = LabelEncoder()
y_meta = le.fit_transform(meta_df['best_selector'].values)

print(f"Meta-dataset: {X_meta.shape[0]} samples × {X_meta.shape[1]} meta-features")
print(f"Target classes: {le.classes_}")

# Train the meta-model on all data
from sklearn.ensemble import RandomForestClassifier as RFC
meta_model = RFC(n_estimators=100, random_state=42)
meta_model.fit(X_meta, y_meta)

# Feature importance of meta-features
meta_fi = pd.Series(meta_model.feature_importances_, index=available_cols)
print("\nTop-5 most predictive meta-features:")
print(meta_fi.sort_values(ascending=False).head(5).to_string())

## 6. Leave-One-Dataset-Out Evaluation

We evaluate the meta-recommender using LODO cross-validation: train on all but one dataset, predict for the held-out one. This measures true generalisation to unseen datasets.

In [ ]:
n_datasets = len(meta_df)
correct_top1 = []
correct_top3 = []
rank_of_best = []
predictions = []

for held_out_idx in range(n_datasets):
    train_mask = np.ones(n_datasets, dtype=bool)
    train_mask[held_out_idx] = False

    X_train_m = X_meta[train_mask]
    y_train_m = y_meta[train_mask]
    X_test_m = X_meta[~train_mask]
    y_test_m = y_meta[~train_mask]

    if len(np.unique(y_train_m)) < 2:
        continue  # can't train with < 2 classes

    lodo_model = RFC(n_estimators=100, random_state=42)
    lodo_model.fit(X_train_m, y_train_m)

    proba = lodo_model.predict_proba(X_test_m)[0]

    # Map probabilities to all classes (some may be absent in training)
    train_classes = np.unique(y_train_m)
    full_proba = np.zeros(len(le.classes_))
    for i, cls in enumerate(train_classes):
        full_proba[cls] = proba[i]

    ranked_preds = np.argsort(-full_proba)
    true_label = y_test_m[0]
    predicted_top1 = ranked_preds[0]

    is_top1 = predicted_top1 == true_label
    is_top3 = true_label in ranked_preds[:3]
    rank = int(np.where(ranked_preds == true_label)[0][0]) + 1 if true_label in ranked_preds else len(le.classes_)

    correct_top1.append(is_top1)
    correct_top3.append(is_top3)
    rank_of_best.append(rank)

    dataset_name = meta_df.iloc[held_out_idx]['dataset_name']
    true_best = le.classes_[true_label]
    predicted_best = le.classes_[predicted_top1]
    predictions.append({'dataset': dataset_name,
                          'true_best': true_best,
                          'predicted': predicted_best,
                          'correct': is_top1,
                          'rank': rank})

print("Leave-One-Dataset-Out Evaluation Results:")
print(f"  Top-1 accuracy: {np.mean(correct_top1):.3f}  (random baseline: {1/len(le.classes_):.3f})")
print(f"  Top-3 accuracy: {np.mean(correct_top3):.3f}  (random baseline: {3/len(le.classes_):.3f})")
print(f"  Avg rank of best: {np.mean(rank_of_best):.2f}  (random baseline: {(len(le.classes_)+1)/2:.2f})")
print(f"  N evaluations: {len(correct_top1)}")

pred_df = pd.DataFrame(predictions)
print("\nPer-dataset predictions:")
print(pred_df.to_string(index=False))

## 7. Visualise Meta-Feature Importance and Recommendations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Plot 1: Meta-feature importance ---
fi_sorted = meta_fi.sort_values(ascending=True)
colours = plt.cm.RdYlGn(fi_sorted.values / fi_sorted.max())
axes[0].barh(fi_sorted.index, fi_sorted.values, color=colours, edgecolor='black', linewidth=0.6)
axes[0].set_xlabel('Feature Importance', fontsize=11)
axes[0].set_title('Meta-Feature Importance\n(Which dataset properties predict best selector?)',
                   fontsize=11)

# --- Plot 2: Confusion matrix for predictions ---
all_selectors = le.classes_
conf_matrix = np.zeros((len(all_selectors), len(all_selectors)), dtype=int)
for p_item in predictions:
    true_idx = np.where(all_selectors == p_item['true_best'])[0]
    pred_idx = np.where(all_selectors == p_item['predicted'])[0]
    if len(true_idx) > 0 and len(pred_idx) > 0:
        conf_matrix[true_idx[0], pred_idx[0]] += 1

im = axes[1].imshow(conf_matrix, cmap='Blues')
axes[1].set_xticks(range(len(all_selectors)))
axes[1].set_yticks(range(len(all_selectors)))
axes[1].set_xticklabels(all_selectors, rotation=30, ha='right', fontsize=9)
axes[1].set_yticklabels(all_selectors, fontsize=9)
axes[1].set_xlabel('Predicted Best Selector', fontsize=10)
axes[1].set_ylabel('True Best Selector', fontsize=10)
axes[1].set_title('Confusion Matrix: LODO Predictions\n(Diagonal = correct recommendations)',
                   fontsize=11)
plt.colorbar(im, ax=axes[1])

for i in range(len(all_selectors)):
    for j in range(len(all_selectors)):
        if conf_matrix[i, j] > 0:
            axes[1].text(j, i, conf_matrix[i, j], ha='center', va='center',
                          fontsize=11, color='white' if conf_matrix[i, j] > 1 else 'black')

plt.tight_layout()
plt.savefig('meta_learner_results.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. The Practical Recommender Tool

Build the final production-ready recommender: give it a new dataset, get a ranked list of selector recommendations.

In [ ]:
class FeatureSelectionRecommender:
    """
    Trained meta-learning recommender for feature selection.
    Input: a new (X, y) dataset.
    Output: ranked list of recommended feature selectors with confidence scores.
    """

    def __init__(self, meta_model, label_encoder, feature_cols):
        self.meta_model = meta_model
        self.le = label_encoder
        self.feature_cols = feature_cols

    def recommend(self, X: np.ndarray, y: np.ndarray,
                   top_n: int = 3) -> pd.DataFrame:
        """
        Recommend top-n selectors for a new dataset.

        Returns DataFrame with selector name, confidence, and rationale.
        """
        mf = compute_meta_features(X, y, name='new_dataset')
        x_vec = np.array([mf.get(c, 0.0) for c in self.feature_cols]).reshape(1, -1)

        proba = self.meta_model.predict_proba(x_vec)[0]
        recommendations = sorted(
            zip(self.le.classes_, proba),
            key=lambda x: x[1], reverse=True
        )[:top_n]

        # Build explanation from key meta-features
        rationales = {
            'mi': 'Fast, handles nonlinear relationships, good for independent features',
            'lasso': 'Sparse linear selection, good when true model is linear',
            'rf_importance': 'Captures interactions, robust to correlated features',
            'rf_boruta_approx': 'Principled all-relevant selection, good with many noise features',
            'elastic_net': 'Handles correlated features better than LASSO',
        }

        rows = []
        for sel_name, conf in recommendations:
            rows.append({
                'Selector': sel_name,
                'Confidence': f'{conf:.1%}',
                'Rationale': rationales.get(sel_name, ''),
            })

        # Print key meta-features for transparency
        print(f"Dataset meta-features:")
        print(f"  n={int(mf.get('n_samples', 0))}, p={int(mf.get('n_features', 0))}, "
              f"n/p={np.exp(mf.get('log_ratio_n_p', 0)) - 1:.1f}")
        print(f"  Mean abs correlation: {mf.get('mean_abs_correlation', 0):.3f}")
        print(f"  Fraction zero-MI features: {mf.get('frac_zero_mi', 0):.3f}")
        print(f"  Decision stump lift: {mf.get('dt1_lift', 0):.3f}")
        print()

        return pd.DataFrame(rows)


recommender = FeatureSelectionRecommender(meta_model, le, available_cols)

# Test on a new dataset: synthetic high-dimensional noisy
print("=" * 60)
print("DEMO: Recommending selector for a new high-dimensional dataset")
print("=" * 60)
X_new, y_new = make_classification(
    n_samples=400, n_features=150, n_informative=8,
    n_redundant=5, flip_y=0.05, random_state=99
)
recs = recommender.recommend(X_new, y_new, top_n=3)
print("Top-3 recommendations:")
print(recs.to_string(index=False))

print()
print("=" * 60)
print("DEMO: Recommending selector for a highly correlated dataset")
print("=" * 60)
X_corr, y_corr = make_classification(
    n_samples=500, n_features=25, n_informative=6,
    n_redundant=16, flip_y=0.03, random_state=88
)
recs_corr = recommender.recommend(X_corr, y_corr, top_n=3)
print("Top-3 recommendations:")
print(recs_corr.to_string(index=False))

## Exercise 9.1: Add a New Dataset to the Meta-Training Set

**Task:** Add the `load_digits()` dataset with the **first 20 features only** as a new entry to the meta-training set. Retrain the meta-model and check if the LODO accuracy improves or changes.

**Steps:**
1. Create a new dataset dict with `X = load_digits().data[:, :20]`
2. Compute its meta-features using `compute_meta_features`
3. Run all 5 selectors and record the best
4. Append to `meta_df` and retrain the meta-model
5. Re-evaluate with LODO

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Load digits dataset and take first 20 features
# 2. Compute meta-features
# 3. Evaluate 5 selectors
# 4. Add row to meta_df
# 5. Retrain meta_model
# 6. Print new LODO accuracy

print("TODO: Implement this exercise")

In [ ]:
# SOLUTION
# --------
from sklearn.datasets import load_digits as _load_digits

digits_all = _load_digits()
X_new_ds = digits_all.data[:, :20]   # first 20 features
y_new_ds = digits_all.target

# Step 1: meta-features
new_mf = compute_meta_features(X_new_ds, y_new_ds, name='digits_20feat')

# Step 2: evaluate 5 selectors
new_sel_scores = {}
for sel_name in SELECTOR_NAMES:
    sel = run_selector(sel_name, X_new_ds, y_new_ds, n_select=10)
    s = cross_val_score(
        GradientBoostingClassifier(n_estimators=80, random_state=42),
        X_new_ds[:, sel], y_new_ds,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='balanced_accuracy'
    ).mean()
    new_sel_scores[sel_name] = s

new_mf['best_selector'] = max(new_sel_scores, key=new_sel_scores.get)
new_mf['best_score'] = max(new_sel_scores.values())
for sel_name, score in new_sel_scores.items():
    new_mf[f'score_{sel_name}'] = score

print(f"New dataset best selector: {new_mf['best_selector']} "
      f"(score={new_mf['best_score']:.4f})")

# Step 3: append to meta_df and retrain
meta_df_extended = pd.concat([meta_df, pd.DataFrame([new_mf])], ignore_index=True)

X_meta_ext = meta_df_extended[available_cols].fillna(0).values
y_meta_ext = le.transform(
    meta_df_extended['best_selector'].fillna('mi').values  # fill NaN with fallback
)

meta_model_ext = RFC(n_estimators=100, random_state=42)
meta_model_ext.fit(X_meta_ext, y_meta_ext)

# Quick accuracy check (in-sample, not LODO — but shows the model updated)
in_sample_acc = (meta_model_ext.predict(X_meta_ext) == y_meta_ext).mean()
print(f"\nExtended meta-dataset: {len(meta_df_extended)} datasets")
print(f"In-sample accuracy (extended meta-model): {in_sample_acc:.3f}")
print("Test passed: new dataset added and model retrained successfully.")

## Summary

### Key Takeaways

1. **Meta-features capture dataset personality:** Correlation structure, MI distribution, landmarking accuracy, and n/p ratio collectively characterise why one selector works better than another on a specific dataset.

2. **Even small meta-training sets are informative:** With only 12 datasets, our meta-recommender achieves better-than-random accuracy. With 50+ datasets, practical systems achieve 50–65% top-1 accuracy.

3. **LODO is the correct evaluation protocol:** It measures true generalisation to unseen datasets, not just different samples from the same distribution.

4. **The recommender provides a practical starting point:** Use the top-3 recommendations as the starting point for your selector search, then validate with cross-validation.

5. **Meta-learning reduces exploration cost:** Instead of running 5 selectors and picking the best (5× cost), the recommender suggests 2–3 and you validate those (2–3× cost).

### What's Next
Module 11 covers production feature selection: deploying selection pipelines, monitoring feature drift, and maintaining selection quality over time.

### Additional Resources
- Guide 03: `guides/03_meta_learning_guide.md`
- Vanschoren (2019): Meta-learning: A survey — comprehensive overview
- Feurer et al. (2015): Auto-sklearn with meta-learning warm-start — practical implementation